In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph.message import add_messages

In [2]:
load_dotenv()  # Load environment variables from .env file
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
from langchain_classic.prompts import MessagesPlaceholder


prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Reply briefly when possible."
    ),
    MessagesPlaceholder(variable_name="messages"),
    ])

In [9]:
MAX_INPUT_TOKENS = 200  # Define a maximum token limit for input messages

In [10]:
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

def chat_trim_node(state: MessagesState):
    messages = state["messages"]
    trimmed_messages = trim_messages(messages = messages, max_tokens=MAX_INPUT_TOKENS, token_counter=count_tokens_approximately, strategy="last")
    
    print(f"Approx count of tokens in messages: {count_tokens_approximately(trimmed_messages)}")
    chain = prompt_template | llm
    response = chain.invoke({"messages": trimmed_messages})
    
    return {"messages": [response]}

In [11]:
graph = StateGraph(MessagesState)

In [12]:
graph.add_node('chat_node', chat_trim_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [13]:
from langgraph.checkpoint.memory import MemorySaver

workflow = graph.compile(MemorySaver())
config_t2 = {"configurable": {"thread_id": "user_2"}}

In [14]:
workflow.invoke({"messages": [{"role": "user", "content": "Hello, how are you?"}]}, config=config_t2)
workflow.invoke({"messages": [{"role": "user", "content": "What is the weather like today?"}]}, config=config_t2)
workflow.invoke({"messages": [{"role": "user", "content": "Tell me a joke."}]}, config=config_t2)

Approx count of tokens in messages: 9
Approx count of tokens in messages: 41
Approx count of tokens in messages: 81


{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='96a5277c-ae6a-4aa3-983d-8ac6f2591ad2'),
  AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you. How can I help you today?", 'extras': {'signature': 'El4KXAFpFH0TKn6dgS1l8jWtmv0gCM5saqjpWGFOpne9BOZqhaLf7215YNZBU5C3fUdYSlnLrV+4QSguvLskmMoLYqi6xZQCQoohBkIHTYaskhp20MSN8V+Y8HBeld3u'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cf61-884d-7f60-b39c-de67260fbeea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 18, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is the weather like today?', additional_kwargs={}, response_metadata={}, id='8efb0d9d-7ebc-42b7-b9a9-10b3abb1ffe4'),
  AIMessage(content=[{'type': 'text', 'text': "I don't have access

In [ ]:
workflow.